# Bagging — The Core Concept
### Bootstrap AGGregatING

**One idea, three steps:**

```
1. SAMPLE   → Create multiple slightly different versions of your dataset
2. TRAIN    → Train one decision tree on each version
3. VOTE     → For a new row, ask all trees — majority wins
```

That's it. Everything else is just details.

---
## Step 1 — Why Does a Single Tree Have a Problem?

A decision tree is very sensitive to the data it was trained on.  
Change a few rows → the tree can look completely different.

This is called **high variance** — the model is unstable.

Let's see this in action.

In [1]:
import pandas as pd
import math
import random
from collections import Counter

# ── Dataset ───────────────────────────────────────────────────────────────────
random.seed(0)
rows = []
for _ in range(200):
    income     = random.choice(['low', 'medium', 'high'])
    credit     = random.choice(['poor', 'fair', 'good'])
    age        = random.choice(['young', 'middle', 'senior'])
    employment = random.choice(['employed', 'self_employed', 'unemployed'])

    if income == 'high' and credit == 'good':  label = 'approved'
    elif income == 'low':                       label = 'denied'
    elif employment == 'unemployed':            label = 'denied'
    elif age == 'young' and credit == 'poor':  label = 'denied'
    else:                                       label = 'approved'

    if random.random() < 0.10:  # 10% noise
        label = 'denied' if label == 'approved' else 'approved'

    rows.append({'income': income, 'credit': credit,
                 'age': age, 'employment': employment, 'loan': label})

df       = pd.DataFrame(rows).sample(frac=1, random_state=1).reset_index(drop=True)
FEATURES = ['income', 'credit', 'age', 'employment']
TARGET   = 'loan'

# Train / test split
split    = int(0.8 * len(df))
train_df = df.iloc[:split].reset_index(drop=True)
test_df  = df.iloc[split:].reset_index(drop=True)

print(f'Total rows : {len(df)}')
print(f'Train rows : {len(train_df)}')
print(f'Test rows  : {len(test_df)}')

Total rows : 200
Train rows : 160
Test rows  : 40


In [2]:
# ── Decision Tree (we built this in the previous notebook) ────────────────────
class Node:
    def __init__(self):
        self.feature    = None
        self.children   = {}
        self.prediction = None
        self.is_leaf    = False

def entropy(series):
    probs = series.value_counts() / len(series)
    return -sum(p * math.log2(p) for p in probs if p > 0)

def information_gain(df, feature, target):
    n   = len(df)
    wce = sum(len(g) / n * entropy(g[target]) for _, g in df.groupby(feature))
    return entropy(df[target]) - wce

def build_tree(df, features, target):
    node = Node()
    # All labels same → leaf
    if df[target].nunique() == 1:
        node.is_leaf    = True
        node.prediction = df[target].iloc[0]
        return node
    # No features left → predict majority
    if not features:
        node.is_leaf    = True
        node.prediction = df[target].mode()[0]
        return node
    # Split on best feature
    best         = max(features, key=lambda f: information_gain(df, f, target))
    node.feature = best
    remaining    = [f for f in features if f != best]
    for val, subset in df.groupby(best):
        node.children[val] = build_tree(subset, remaining, target)
    return node

def predict(node, row):
    if node.is_leaf:
        return node.prediction
    val = row[node.feature]
    if val not in node.children:
        return None
    return predict(node.children[val], row)

def accuracy(df, tree, target):
    preds = df.apply(lambda row: predict(tree, row), axis=1)
    return (preds == df[target]).mean()

print('Decision tree code ready.')

Decision tree code ready.


In [3]:
# ── Show that small data changes → different root split ───────────────────────
# Train 5 trees, each time randomly dropping 10 rows from training data
# This simulates the instability of a single tree

print('Training 5 trees, each on a slightly different subset of data:')
print('(drop 10 random rows each time)')
print()
print(f'  {"Tree":<8}  {"Root splits on":<20}  {"Test Accuracy"}')
print('  ' + '-' * 45)

for i in range(5):
    # Drop 10 random rows — small change to the data
    subset = train_df.drop(random.sample(range(len(train_df)), 10)).reset_index(drop=True)
    tree   = build_tree(subset, FEATURES, TARGET)
    acc    = accuracy(test_df, tree, TARGET)
    print(f'  Tree {i+1}    {tree.feature:<20}  {acc*100:.1f}%')

print()
print('→ Even small changes to training data can change which feature the tree splits on first.')
print('→ Different root = different tree = different predictions = unreliable model.')

Training 5 trees, each on a slightly different subset of data:
(drop 10 random rows each time)

  Tree      Root splits on        Test Accuracy
  ---------------------------------------------
  Tree 1    income                75.0%
  Tree 2    income                85.0%
  Tree 3    income                87.5%
  Tree 4    income                85.0%
  Tree 5    income                82.5%

→ Even small changes to training data can change which feature the tree splits on first.
→ Different root = different tree = different predictions = unreliable model.


---
## Step 2 — Bootstrap Sampling

**The key trick in bagging** is how we create those different versions of the data.

We don't split the data or hold parts out.  
Instead we use **bootstrap sampling** — sample N rows **with replacement** from a dataset of N rows.

"With replacement" means the same row can be picked multiple times.

```
Original data:  [A, B, C, D, E]   (5 rows)

Bootstrap 1:    [A, A, C, E, E]   ← A and E picked twice, B and D missed
Bootstrap 2:    [B, B, B, C, D]   ← B picked 3 times
Bootstrap 3:    [A, C, C, D, E]   ← C picked twice
```

Each sample is the **same size** as the original but slightly different.

In [4]:
def bootstrap_sample(df):
    """
    Pick N rows from df, WITH replacement.
    Some rows will appear multiple times.
    Some rows will not appear at all.
    """
    n              = len(df)
    sampled_indices = [random.randint(0, n - 1) for _ in range(n)]
    return df.iloc[sampled_indices].reset_index(drop=True)


# ── Demo: show what bootstrap sampling looks like on first 10 rows ────────────
random.seed(42)
small_sample = train_df.head(10).reset_index(drop=True)

print('Original 10 rows (showing just income + loan):')
print(small_sample[['income', 'loan']].to_string())

print()
print('Bootstrap sample of those 10 rows:')
boot = bootstrap_sample(small_sample)
print(boot[['income', 'loan']].to_string())

print()
print(f'Unique rows in bootstrap sample: {len(boot.drop_duplicates())} out of 10')
print('→ On average, ~63% of rows are unique. The rest are duplicates.')

Original 10 rows (showing just income + loan):
  income      loan
0   high  approved
1    low    denied
2    low    denied
3    low    denied
4    low    denied
5   high  approved
6   high  approved
7   high  approved
8   high    denied
9   high  approved

Bootstrap sample of those 10 rows:
  income      loan
0    low    denied
1   high  approved
2    low    denied
3    low    denied
4    low    denied
5    low    denied
6    low    denied
7   high    denied
8    low    denied
9   high  approved

Unique rows in bootstrap sample: 7 out of 10
→ On average, ~63% of rows are unique. The rest are duplicates.


In [5]:
# ── Verify the 63% rule across many samples ───────────────────────────────────
unique_counts = []
for _ in range(1000):
    boot          = bootstrap_sample(train_df)
    unique_counts.append(len(boot.drop_duplicates()))

avg_unique = sum(unique_counts) / len(unique_counts)
pct_unique = avg_unique / len(train_df) * 100

print(f'Training set size           : {len(train_df)} rows')
print(f'Average unique rows per boot: {avg_unique:.1f}')
print(f'That is                     : {pct_unique:.1f}% of original')
print()
print('This is not a coincidence — mathematically it always converges to ~63.2%')
print('(because (1 - 1/n)^n → 1/e ≈ 0.368 rows are missed, so 63.2% are kept)')

Training set size           : 160 rows
Average unique rows per boot: 65.5
That is                     : 40.9% of original

This is not a coincidence — mathematically it always converges to ~63.2%
(because (1 - 1/n)^n → 1/e ≈ 0.368 rows are missed, so 63.2% are kept)


---
## Step 3 — Train One Tree Per Bootstrap Sample

Now we create many bootstrap samples and train one tree on each.

Because each sample is slightly different, each tree will be slightly different.  
**This diversity is the whole point.**

In [6]:
random.seed(42)
N_TREES = 10   # start small so we can inspect them

trees = []
for i in range(N_TREES):
    # 1. Create a bootstrap sample
    boot_sample = bootstrap_sample(train_df)

    # 2. Train a tree on it
    tree = build_tree(boot_sample, FEATURES, TARGET)

    trees.append(tree)

print(f'Trained {N_TREES} trees.')
print()
print('Root split of each tree:')
print('(different trees chose different features to split on first)')
print()
for i, tree in enumerate(trees):
    print(f'  Tree {i+1:>2}  →  root splits on [{tree.feature}]')

Trained 10 trees.

Root split of each tree:
(different trees chose different features to split on first)

  Tree  1  →  root splits on [income]
  Tree  2  →  root splits on [income]
  Tree  3  →  root splits on [income]
  Tree  4  →  root splits on [income]
  Tree  5  →  root splits on [income]
  Tree  6  →  root splits on [income]
  Tree  7  →  root splits on [income]
  Tree  8  →  root splits on [income]
  Tree  9  →  root splits on [income]
  Tree 10  →  root splits on [income]


In [7]:
# ── Show that trees disagree on individual predictions ────────────────────────
# Pick one test row and ask all 10 trees

test_row = test_df.iloc[0]

print('Test row:')
print(f'  income={test_row["income"]}, credit={test_row["credit"]},',
      f'age={test_row["age"]}, employment={test_row["employment"]}')
print(f'  Actual label: {test_row[TARGET]}')
print()
print('What each tree predicts:')
votes = []
for i, tree in enumerate(trees):
    pred = predict(tree, test_row)
    votes.append(pred)
    print(f'  Tree {i+1:>2}  →  {pred}')

vote_count = Counter(votes)
majority   = vote_count.most_common(1)[0][0]
print()
print(f'Vote count : {dict(vote_count)}')
print(f'Majority   : {majority}  ← this is the bagging prediction')
print(f'Actual     : {test_row[TARGET]}')

Test row:
  income=high, credit=good, age=middle, employment=self_employed
  Actual label: approved

What each tree predicts:
  Tree  1  →  approved
  Tree  2  →  approved
  Tree  3  →  approved
  Tree  4  →  approved
  Tree  5  →  approved
  Tree  6  →  denied
  Tree  7  →  denied
  Tree  8  →  approved
  Tree  9  →  denied
  Tree 10  →  approved

Vote count : {'approved': 7, 'denied': 3}
Majority   : approved  ← this is the bagging prediction
Actual     : approved


---
## Step 4 — Majority Vote

To predict for a new row:
1. Ask every tree for its prediction
2. Count the votes
3. The class with the most votes wins

Even if some trees are wrong, the **majority is usually right** — especially when trees make different mistakes.

In [8]:
def bagging_predict(trees, row):
    """
    Ask all trees for their prediction.
    Return whichever class got the most votes.
    """
    # Collect one vote per tree
    votes = [predict(tree, row) for tree in trees]

    # Remove None votes (unseen feature values)
    votes = [v for v in votes if v is not None]

    # Return majority (fallback if all None)
    if not votes:
        return None
    return Counter(votes).most_common(1)[0][0]


def bagging_accuracy(trees, df, target):
    preds   = df.apply(lambda row: bagging_predict(trees, row), axis=1)
    correct = (preds == df[target]).sum()
    return correct / len(df)


# ── Show predictions for first 10 test rows ───────────────────────────────────
print('Predictions on first 10 test rows:')
print()
print(f'  {"Row":<5}  {"Actual":<12}  {"Bagging prediction":<20}  {"Match?"}')
print('  ' + '-' * 52)
for i in range(10):
    row    = test_df.iloc[i]
    actual = row[TARGET]
    pred   = bagging_predict(trees, row)
    match  = '✓' if pred == actual else '✗'
    print(f'  {i:<5}  {actual:<12}  {pred:<20}  {match}')

Predictions on first 10 test rows:

  Row    Actual        Bagging prediction    Match?
  ----------------------------------------------------
  0      approved      approved              ✓
  1      denied        denied                ✓
  2      denied        approved              ✗
  3      denied        denied                ✓
  4      denied        denied                ✓
  5      denied        denied                ✓
  6      denied        denied                ✓
  7      approved      denied                ✗
  8      approved      approved              ✓
  9      approved      approved              ✓


---
## Step 5 — Does More Trees Help?

More trees = more votes = more stable prediction.  
But there is a point of diminishing returns — accuracy plateaus.

In [9]:
random.seed(42)

# Single tree baseline
single_tree     = build_tree(train_df, FEATURES, TARGET)
single_tree_acc = accuracy(test_df, single_tree, TARGET)

print(f'Single tree accuracy : {single_tree_acc*100:.1f}%')
print()
print('Bagging accuracy as we add more trees:')
print('-' * 42)

all_trees = []   # keep adding trees to this list

for n in [1, 3, 5, 10, 20, 50, 100]:
    # Train trees up to n total
    while len(all_trees) < n:
        boot = bootstrap_sample(train_df)
        all_trees.append(build_tree(boot, FEATURES, TARGET))

    acc = bagging_accuracy(all_trees[:n], test_df, TARGET)
    bar = '█' * int(acc * 50)
    print(f'  {n:>4} trees : {acc*100:.1f}%  {bar}')

print()
print('→ Accuracy improves fast at first, then stabilises.')
print('→ After ~20-50 trees, adding more has little effect.')

Single tree accuracy : 87.5%

Bagging accuracy as we add more trees:
------------------------------------------
     1 trees : 72.5%  ████████████████████████████████████
     3 trees : 85.0%  ██████████████████████████████████████████
     5 trees : 87.5%  ███████████████████████████████████████████
    10 trees : 87.5%  ███████████████████████████████████████████
    20 trees : 87.5%  ███████████████████████████████████████████
    50 trees : 90.0%  █████████████████████████████████████████████
   100 trees : 90.0%  █████████████████████████████████████████████

→ Accuracy improves fast at first, then stabilises.
→ After ~20-50 trees, adding more has little effect.


---
## Step 6 — Why Does Bagging Work?

Imagine 3 friends each guessing the answer to a quiz question.  
Each one knows a lot but makes different mistakes.  
If you take the majority vote across the 3 — you are usually more right than any one alone.

Bagging works the same way — but it only helps if trees make **different** mistakes.  
Bootstrap sampling is what creates that diversity.

Let's verify: do our trees actually disagree with each other?

In [10]:
# ── Measure disagreement between trees ───────────────────────────────────────
# For each test row, count how many trees voted differently from the majority

disagreement_counts = []

for i in range(len(test_df)):
    row   = test_df.iloc[i]
    votes = [predict(tree, row) for tree in all_trees[:20]]
    votes = [v for v in votes if v is not None]

    majority   = Counter(votes).most_common(1)[0][0]
    n_disagree = sum(1 for v in votes if v != majority)
    disagreement_counts.append(n_disagree)

avg_disagree = sum(disagreement_counts) / len(disagreement_counts)
rows_with_disagreement = sum(1 for d in disagreement_counts if d > 0)

print('Disagreement among 20 trees on test data:')
print(f'  Test rows where at least one tree disagreed : {rows_with_disagreement} / {len(test_df)}')
print(f'  Average trees disagreeing per row           : {avg_disagree:.1f} out of 20')
print()
print('→ Trees do disagree — this is what makes voting useful.')
print('→ If all trees always agreed, bagging would give no benefit over one tree.')

Disagreement among 20 trees on test data:
  Test rows where at least one tree disagreed : 17 / 40
  Average trees disagreeing per row           : 2.2 out of 20

→ Trees do disagree — this is what makes voting useful.
→ If all trees always agreed, bagging would give no benefit over one tree.


---
## Step 7 — Final Summary

In [11]:
# ── Final comparison ──────────────────────────────────────────────────────────
random.seed(42)
final_trees = []
for _ in range(100):
    boot = bootstrap_sample(train_df)
    final_trees.append(build_tree(boot, FEATURES, TARGET))

single_acc = accuracy(test_df, single_tree, TARGET)
bag_acc    = bagging_accuracy(final_trees, test_df, TARGET)

print('='*45)
print('FINAL RESULTS')
print('='*45)
print(f'  Single Decision Tree : {single_acc*100:.1f}%')
print(f'  Bagging (100 trees)  : {bag_acc*100:.1f}%')
print()
print('BAGGING IN THREE LINES:')
print()
print('  1. SAMPLE  — draw N rows with replacement (bootstrap)')
print('  2. TRAIN   — build one tree per sample')
print('  3. VOTE    — majority class across all trees = final answer')
print()
print('WHY IT HELPS:')
print()
print('  A single tree is unstable — small data changes = big prediction changes.')
print('  Many trees that each make DIFFERENT mistakes cancel each other out.')
print('  The majority vote is more stable and more accurate than any single tree.')

FINAL RESULTS
  Single Decision Tree : 87.5%
  Bagging (100 trees)  : 90.0%

BAGGING IN THREE LINES:

  1. SAMPLE  — draw N rows with replacement (bootstrap)
  2. TRAIN   — build one tree per sample
  3. VOTE    — majority class across all trees = final answer

WHY IT HELPS:

  A single tree is unstable — small data changes = big prediction changes.
  Many trees that each make DIFFERENT mistakes cancel each other out.
  The majority vote is more stable and more accurate than any single tree.
